## Importing Libraries | Reading Tables 

In [1]:
from tokenize import Ignore
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np 
import os
import warnings
from optbinning import OptimalBinning
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

## Config

In [34]:
ROOT_PATH = r"input_data\row_input_data"

RUN_AGG = False
RUN_MERGE = False

ROW_AGG_SAVE_PATH = r"input_data\row_agg_data"
ROW_MRGD_SAVE_PATH = r"input_data\row_merged_data"
ROW_FS_SAVE_PATH = r"input_data\row_fs_data"

SAVE_FILES = False

In [3]:
%%time

app_train = pd.read_csv(os.path.join(ROOT_PATH, 'application_train.csv'))
app_test = pd.read_csv(os.path.join(ROOT_PATH, 'application_test.csv'))
bureau_bal = pd.read_csv(os.path.join(ROOT_PATH, 'bureau_balance.csv'))
bureau = pd.read_csv(os.path.join(ROOT_PATH, 'bureau.csv'))
cc_bal = pd.read_csv(os.path.join(ROOT_PATH, "credit_card_balance.csv"))
col_desc = pd.read_csv(os.path.join(ROOT_PATH, "HomeCredit_columns_description.csv"), encoding='latin1')
inst_pmt = pd.read_csv(os.path.join(ROOT_PATH, "installments_payments.csv"))
pos_cash = pd.read_csv(os.path.join(ROOT_PATH, "POS_CASH_balance.csv"))
prev_app = pd.read_csv(os.path.join(ROOT_PATH, "previous_application.csv"))
sub = pd.read_csv(os.path.join(ROOT_PATH, "sample_submission.csv"))

data_dict = {
    'application_train': app_train, 
    'application_test' : app_test, 
    'bureau_balance' : bureau_bal, 
    'bureau' : bureau, 
    'credit_card_balance' : cc_bal, 
    'HomeCredit_columns_description' : col_desc, 
    'installments_payments' : inst_pmt, 
    'POS_CASH_balance' : pos_cash, 
    'previous_application' : prev_app,
    'sample_submission' : sub
    }


print("Shape of application_train.csv:              ", app_train.shape)
print("Shape of application_test.csv:               ", app_test.shape)
print("Shape of bureau_balance.csv:                 ", bureau_bal.shape)
print("Shape of bureau.csv:                         ", bureau.shape)
print("Shape of credit_card_balance.csv:            ", cc_bal.shape)
print("Shape of HomeCredit_columns_description.csv: ", col_desc.shape)
print("Shape of installments_payments.csv:          ", inst_pmt.shape)
print("Shape of POS_CASH_balance.csv:               ", pos_cash.shape)
print("Shape of previous_application.csv:           ", prev_app.shape)
print("Shape of sample_submission.csv:              ", sub.shape)

Shape of application_train.csv:               (307511, 122)
Shape of application_test.csv:                (48744, 121)
Shape of bureau_balance.csv:                  (27299925, 3)
Shape of bureau.csv:                          (1716428, 17)
Shape of credit_card_balance.csv:             (3840312, 23)
Shape of HomeCredit_columns_description.csv:  (219, 5)
Shape of installments_payments.csv:           (13605401, 8)
Shape of POS_CASH_balance.csv:                (10001358, 8)
Shape of previous_application.csv:            (1670214, 37)
Shape of sample_submission.csv:               (48744, 2)
CPU times: total: 1min 3s
Wall time: 1min 4s


## Aggregation Techniquest

In [4]:
KEYS = {
    'app_train':['SK_ID_CURR'],
    'app_test':['SK_ID_CURR'],
    'bureau': ['SK_ID_BUREAU', 'SK_ID_CURR'],
    'bureau_bal' :[ 'SK_ID_BUREAU'],
    'cc_bal' : ['SK_ID_PREV', 'SK_ID_CURR'],
    'inst_pmt': ['SK_ID_PREV', 'SK_ID_CURR'],
    'pos_cash' : ['SK_ID_PREV', 'SK_ID_CURR'],
    'prev_app' : ['SK_ID_PREV', 'SK_ID_CURR']
}

DFS = {
    'app_train': app_train,
    'app_test': app_test,
    'bureau': bureau,
    'bureau_bal': bureau_bal,
    'cc_bal': cc_bal,
    'inst_pmt': inst_pmt,
    'pos_cash': pos_cash,
    'prev_app': prev_app
}

print("Unique values for each foreign key:\n")

for df_name, keys in KEYS.items():
    print(f"{df_name}")
    for key in keys:
        print(f"    {key:<15}: Total rows: {DFS[df_name][key].shape[0]} | Unique: {DFS[df_name][key].nunique()} | ")
    print("-" * 50)


print(f"Aggregation techniques for Bureau Data")
bur_cat = bureau.select_dtypes(include='object').columns
AGG_BUREAU = {col: ['mode'] if col in bur_cat else ['mean', 'median', 'max', 'min'] for col in bureau.columns}


print(f"Aggregation techniques for Bureau Balance Data")
bur_bal_cat = bureau_bal.select_dtypes(include='object').columns
AGG_BUREAU_BAL = {col: ['mode'] if col in bur_bal_cat else ['mean', 'median', 'max', 'min'] for col in bureau_bal.columns}


print(f"Aggregation techniques for Credit Card Balance Data")
cc_bal_cat = cc_bal.select_dtypes(include='object').columns
AGG_CC_BAL = {col: ['mode'] if col in cc_bal_cat else ['mean', 'median', 'max', 'min'] for col in cc_bal.columns}


print(f"Aggregation techniques for Installment Payments Data")
inst_pmt_cat = inst_pmt.select_dtypes(include='object').columns
AGG_INST_PMT = {col: ['mode'] if col in inst_pmt_cat else ['mean', 'median', 'max', 'min'] for col in inst_pmt.columns}


print(f"Aggregation techniques for POS Cash Balance Data")
pos_cash_cat = pos_cash.select_dtypes(include='object').columns
AGG_POS_CASH = {col: ['mode'] if col in pos_cash_cat else ['mean', 'median', 'max', 'min'] for col in pos_cash.columns}


print(f"Aggregation techniques for Previous Application Data")
prev_app_cat = prev_app.select_dtypes(include='object').columns
AGG_PREV_APP = {col: ['mode'] if col in prev_app_cat else ['mean', 'median', 'max', 'min'] for col in prev_app.columns}

Unique values for each foreign key:

app_train
    SK_ID_CURR     : Total rows: 307511 | Unique: 307511 | 
--------------------------------------------------
app_test
    SK_ID_CURR     : Total rows: 48744 | Unique: 48744 | 
--------------------------------------------------
bureau
    SK_ID_BUREAU   : Total rows: 1716428 | Unique: 1716428 | 
    SK_ID_CURR     : Total rows: 1716428 | Unique: 305811 | 
--------------------------------------------------
bureau_bal
    SK_ID_BUREAU   : Total rows: 27299925 | Unique: 817395 | 
--------------------------------------------------
cc_bal
    SK_ID_PREV     : Total rows: 3840312 | Unique: 104307 | 
    SK_ID_CURR     : Total rows: 3840312 | Unique: 103558 | 
--------------------------------------------------
inst_pmt
    SK_ID_PREV     : Total rows: 13605401 | Unique: 997752 | 
    SK_ID_CURR     : Total rows: 13605401 | Unique: 339587 | 
--------------------------------------------------
pos_cash
    SK_ID_PREV     : Total rows: 10001358 | Un

In [ ]:
%%time
def get_mode(x):
    x = x.dropna()
    if len(x) == 0:
        return np.nan
    mode_val = x.mode()
    if len(mode_val) == 0:
        return np.nan
    return mode_val.iloc[0]


def aggregate_dataset(df, agg_dict):
    groupby_col = list(agg_dict.keys())[0]
    agg_func = {}
    for col, methods in agg_dict.items():
        if col == groupby_col:
            continue

        agg_func[col] = [
            get_mode if method == 'mode' else method
            for method in methods
        ]

    agg_df = df.groupby(groupby_col).agg(agg_func)
    agg_df.columns = [
        f"{col}_{func if isinstance(func, str) else 'mode'}".upper()
        for col, func in agg_df.columns
    ]
    agg_df.reset_index(inplace=True)
    return agg_df


if RUN_AGG:
    bureau_agg = aggregate_dataset(bureau, AGG_BUREAU)

    print(f"bureau_agg Shape        : {bureau_agg.shape}")
    print(f"bureau_agg Null Values  : {bureau_agg.isnull().sum().sum()}")
    print(f"bureau_agg Duplicates   : {bureau_agg.duplicated().sum()}")
    print("-" * 60)

    bureau_bal_agg = aggregate_dataset(bureau_bal, AGG_BUREAU_BAL)

    print(f"bureau_bal_agg Shape       : {bureau_bal_agg.shape}")
    print(f"bureau_bal_agg Null Values : {bureau_bal_agg.isnull().sum().sum()}")
    print(f"bureau_bal_agg Duplicates  : {bureau_bal_agg.duplicated().sum()}")
    print("-" * 60)

    cc_bal_agg = aggregate_dataset(cc_bal, AGG_CC_BAL)

    print(f"cc_bal_agg Shape        : {cc_bal_agg.shape}")
    print(f"cc_bal_agg Null Values  : {cc_bal_agg.isnull().sum().sum()}")
    print(f"cc_bal_agg Duplicates   : {cc_bal_agg.duplicated().sum()}")
    print("-" * 60)

    inst_pmt_agg = aggregate_dataset(inst_pmt, AGG_INST_PMT)

    print(f"inst_pmt_agg Shape        : {inst_pmt_agg.shape}")
    print(f"inst_pmt_agg Null Values  : {inst_pmt_agg.isnull().sum().sum()}")
    print(f"inst_pmt_agg Duplicates   : {inst_pmt_agg.duplicated().sum()}")
    print("-" * 60)

    pos_cash_agg = aggregate_dataset(pos_cash, AGG_POS_CASH)

    print(f"pos_cash_agg Shape        : {pos_cash_agg.shape}")
    print(f"pos_cash_agg Null Values  : {pos_cash_agg.isnull().sum().sum()}")
    print(f"pos_cash_agg Duplicates   : {pos_cash_agg.duplicated().sum()}")
    print("-" * 60)

    prev_app_agg = aggregate_dataset(prev_app, AGG_PREV_APP)

    print(f"prev_app_agg Shape        : {prev_app_agg.shape}")
    print(f"prev_app_agg Null Values  : {prev_app_agg.isnull().sum().sum()}")
    print(f"prev_app_agg Duplicates   : {prev_app_agg.duplicated().sum()}")
    print("-" * 60)

    if SAVE_FILES:
        os.makedirs(ROW_AGG_SAVE_PATH, exist_ok=True)

        bureau_agg.to_csv(fr"{ROW_AGG_SAVE_PATH}\bureau_agg.csv", index=False)
        bureau_bal_agg.to_csv(fr"{ROW_AGG_SAVE_PATH}\bureau_bal_agg.csv", index=False)
        cc_bal_agg.to_csv(fr"{ROW_AGG_SAVE_PATH}\cc_bal_agg.csv", index=False)
        inst_pmt_agg.to_csv(fr"{ROW_AGG_SAVE_PATH}\inst_pmt_agg.csv", index=False)
        pos_cash_agg.to_csv(fr"{ROW_AGG_SAVE_PATH}\pos_cash_agg.csv", index=False)
        prev_app_agg.to_csv(fr"{ROW_AGG_SAVE_PATH}\prev_app_agg.csv", index=False)

        print("All aggregated dataframes saved successfully.")

CPU times: total: 0 ns
Wall time: 7.15 μs


## Table Mering | Feature Selection

In [6]:
ROW_AGG_SAVE_PATH = r"input_data\row_agg_data"

bureau_agg = pd.read_csv(f"{ROW_AGG_SAVE_PATH}\\bureau_agg.csv")
bureau_bal_agg = pd.read_csv(f"{ROW_AGG_SAVE_PATH}\\bureau_bal_agg.csv")
cc_bal_agg = pd.read_csv(f"{ROW_AGG_SAVE_PATH}\\cc_bal_agg.csv")
inst_pmt_agg = pd.read_csv(f"{ROW_AGG_SAVE_PATH}\\inst_pmt_agg.csv")
pos_cash_agg = pd.read_csv(f"{ROW_AGG_SAVE_PATH}\\pos_cash_agg.csv")
prev_app_agg = pd.read_csv(f"{ROW_AGG_SAVE_PATH}\\prev_app_agg.csv")

In [ ]:
PRIMARY_JOIN_KEY = 'SK_ID_CURR'

AGG_TO_APP_KEYS = {
    'bureau_agg': 'SK_ID_CURR',
    'bureau_bal_agg': 'SK_ID_BUREAU',
    'cc_bal_agg': 'SK_ID_PREV',
    'inst_pmt_agg': 'SK_ID_PREV',
    'pos_cash_agg': 'SK_ID_PREV',
    'prev_app_agg': 'SK_ID_PREV',
}


def _second_level_agg(df, group_col, feature_cols):
    numeric = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    non_numeric = [c for c in feature_cols if c not in numeric]

    out = df.groupby(group_col, as_index=False)[numeric].mean()
    if non_numeric:
        mode_part = (
            df.groupby(group_col)[non_numeric]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
            .reset_index()
        )
        out = out.merge(mode_part, on=group_col, how='left')
    return out


def prepare_agg_for_app(agg_df, agg_name, bureau_df=None):
    join_key = AGG_TO_APP_KEYS[agg_name]

    if join_key == PRIMARY_JOIN_KEY:
        return agg_df

    if agg_name == 'bureau_bal_agg':
        bureau_map = bureau_df[['SK_ID_BUREAU', PRIMARY_JOIN_KEY]].drop_duplicates()
        merged = agg_df.merge(bureau_map, on='SK_ID_BUREAU', how='left')
        feature_cols = [c for c in merged.columns if c not in ('SK_ID_BUREAU', PRIMARY_JOIN_KEY)]
        return _second_level_agg(merged.dropna(subset=[PRIMARY_JOIN_KEY]), PRIMARY_JOIN_KEY, feature_cols)

    df = agg_df.copy()
    df[PRIMARY_JOIN_KEY] = df['SK_ID_CURR_MAX']
    drop_cols = {
        'SK_ID_PREV',
        'SK_ID_CURR_MEAN',
        'SK_ID_CURR_MEDIAN',
        'SK_ID_CURR_MAX',
        'SK_ID_CURR_MIN',
    }
    feature_cols = [c for c in df.columns if c not in drop_cols and c != PRIMARY_JOIN_KEY]
    return _second_level_agg(df.dropna(subset=[PRIMARY_JOIN_KEY]), PRIMARY_JOIN_KEY, feature_cols)


def join_agg_to_app(app_df, agg_tables, bureau_df=None):
    merged = app_df.copy()
    for agg_name, agg_df in agg_tables.items():
        print(f"Merging start for {agg_name} ...")
        agg_at_curr = prepare_agg_for_app(agg_df, agg_name, bureau_df=bureau_df)
        suffix = f"_{agg_name.replace('_agg', '')}"
        rename_map = {
            col: f"{col}{suffix}"
            for col in agg_at_curr.columns
            if col != PRIMARY_JOIN_KEY
        }
        agg_at_curr = agg_at_curr.rename(columns=rename_map)
        merged = merged.merge(agg_at_curr, on=PRIMARY_JOIN_KEY, how='left')
    return merged


agg_tables = {
    'bureau_agg': bureau_agg,
    'bureau_bal_agg': bureau_bal_agg,
    'cc_bal_agg': cc_bal_agg,
    'inst_pmt_agg': inst_pmt_agg,
    'pos_cash_agg': pos_cash_agg,
    'prev_app_agg': prev_app_agg,
}

if RUN_MERGE:
    app_train_merged = join_agg_to_app(app_train, agg_tables, bureau_df=bureau)
    app_test_merged = join_agg_to_app(app_test, agg_tables, bureau_df=bureau)

    if SAVE_FILES:
        app_train_merged.to_csv(os.path.join(ROW_MRGD_SAVE_PATH, "app_train_merged.csv"), index=False)
        app_test_merged.to_csv(os.path.join(ROW_MRGD_SAVE_PATH, "app_test_merged.csv"), index=False)

    print(f"app_train shape:        {app_train.shape} -> {app_train_merged.shape}")
    print(f"app_test shape:         {app_test.shape} -> {app_test_merged.shape}")

## 1. Fill Rate Analysis
- Measures missingness
- FR = (Non Missing Records / Total Records)

In [8]:
app_train_merged = pd.read_csv(os.path.join(ROW_MRGD_SAVE_PATH, "app_train_merged.csv"))
app_test_merged = pd.read_csv(os.path.join(ROW_MRGD_SAVE_PATH, "app_test_merged.csv"))
app_train_merged.shape, app_test_merged.shape

((307511, 400), (48744, 399))

In [9]:
FR_THRESHOLD = 5
TARGET = 'TARGET'

fill_rate_report = pd.DataFrame({
    "COLUMN_NAME": app_train_merged.columns,
    "TOTAL_ROWS": len(app_train_merged),
    "NON_NULL_COUNT": app_train_merged.notnull().sum().values,
    "NULL_COUNT": app_train_merged.isnull().sum().values
})

fill_rate_report["FILL_RATE"] = (
    fill_rate_report["NON_NULL_COUNT"]
    / fill_rate_report["TOTAL_ROWS"]
    * 100
).round(2)

fill_rate_report["DROP_FLAG"] = np.where(
    fill_rate_report["FILL_RATE"] < FR_THRESHOLD,
    "DROP",
    "KEEP"
)

fill_rate_report = fill_rate_report.sort_values(
    by="FILL_RATE",
    ascending=True
).reset_index(drop=True)

print(f"Total Features                  : {app_train_merged.shape[1]}")
print(f"Features Below {FR_THRESHOLD}% FR : {(fill_rate_report['DROP_FLAG'] == 'DROP').sum()}")
print(f"Features To Keep                : {(fill_rate_report['DROP_FLAG'] == 'KEEP').sum()}")

fill_rate_report.head(20)

Total Features                  : 400
Features Below 5% FR : 8
Features To Keep                : 392


,COLUMN_NAME,TOTAL_ROWS,NON_NULL_COUNT,NULL_COUNT,FILL_RATE,DROP_FLAG
0,RATE_INTEREST_PRIMARY_MAX_prev_app,307511,4609,302902,1.50,DROP
1,RATE_INTEREST_PRIMARY_MIN_prev_app,307511,4609,302902,1.50,DROP
2,RATE_INTEREST_PRIVILEGED_MEAN_prev_app,307511,4609,302902,1.50,DROP
3,RATE_INTEREST_PRIVILEGED_MEDIAN_prev_app,307511,4609,302902,1.50,DROP
4,RATE_INTEREST_PRIVILEGED_MAX_prev_app,307511,4609,302902,1.50,DROP
5,RATE_INTEREST_PRIVILEGED_MIN_prev_app,307511,4609,302902,1.50,DROP
6,RATE_INTEREST_PRIMARY_MEAN_prev_app,307511,4609,302902,1.50,DROP
7,RATE_INTEREST_PRIMARY_MEDIAN_prev_app,307511,4609,302902,1.50,DROP
8,AMT_PAYMENT_CURRENT_MEDIAN_cc_bal,307511,61060,246451,19.86,KEEP
9,AMT_PAYMENT_CURRENT_MAX_cc_bal,307511,61060,246451,19.86,KEEP


In [10]:
keep_cols = fill_rate_report.loc[
    fill_rate_report["FILL_RATE"] >= FR_THRESHOLD,
    "COLUMN_NAME"
]

app_train_fr = app_train_merged[keep_cols]
app_test_fr = app_test_merged[[col for col in keep_cols.values.tolist() if col != TARGET]]

print(app_train_merged.shape, "->", app_train_fr.shape)
print(app_test_merged.shape, "->", app_test_fr.shape)

(307511, 400) -> (307511, 392)
(48744, 399) -> (48744, 391)


## 2. Zero / Near-Zero Variance Filter
- Remove variables that barely change.

In [11]:
VAR_THRESHOLD = 0.99

nzv_report = []

for col in app_train_fr.columns:

    if col == TARGET:
        continue

    vc = app_train_fr[col].value_counts(dropna=False, normalize=True)

    top_freq = vc.iloc[0] if len(vc) else np.nan

    nzv_report.append({
        "COLUMN_NAME": col,
        "N_UNIQUE": app_train_fr[col].nunique(dropna=False),
        "TOP_FREQ_PCT": round(top_freq * 100, 2),
        "DROP_FLAG": (
            "DROP"
            if (
                app_train_fr[col].nunique(dropna=False) <= 1
                or top_freq >= VAR_THRESHOLD
            )
            else "KEEP"
        )
    })

nzv_report = pd.DataFrame(nzv_report).sort_values(
    "TOP_FREQ_PCT",
    ascending=False
)

print(f"Total Features       : {len(nzv_report)}")
print(f"Near-Zero Variance   : {(nzv_report['DROP_FLAG'] == 'DROP').sum()}")
print(f"Features To Keep     : {(nzv_report['DROP_FLAG'] == 'KEEP').sum()}")

nzv_report.head(20)

Total Features       : 391
Near-Zero Variance   : 18
Features To Keep     : 373


,COLUMN_NAME,N_UNIQUE,TOP_FREQ_PCT,DROP_FLAG
385,FLAG_MOBIL,2,100.00,DROP
348,FLAG_DOCUMENT_12,2,100.00,DROP
349,FLAG_DOCUMENT_10,2,100.00,DROP
362,FLAG_DOCUMENT_2,2,100.00,DROP
360,FLAG_DOCUMENT_4,2,99.99,DROP
364,FLAG_DOCUMENT_7,2,99.98,DROP
343,FLAG_DOCUMENT_21,2,99.97,DROP
358,FLAG_DOCUMENT_17,2,99.97,DROP
356,FLAG_DOCUMENT_20,2,99.95,DROP
355,FLAG_DOCUMENT_19,2,99.94,DROP


In [12]:
keep_cols = nzv_report.loc[
    nzv_report["DROP_FLAG"] == "KEEP",
    "COLUMN_NAME"
]

app_train_nzv = app_train_fr[keep_cols.tolist() + [TARGET]]
app_test_nzv = app_test_fr[keep_cols]

print(app_train_fr.shape, "->", app_train_nzv.shape)
print(app_test_fr.shape, "->", app_test_nzv.shape)

(307511, 392) -> (307511, 374)
(48744, 391) -> (48744, 373)


## 3. WoE / IV Computation

In [13]:
IV_SAMPLE_SIZE = 100000
sample_frac = min(
    1.0,
    IV_SAMPLE_SIZE / len(app_train_nzv)
)

iv_sample, _ = train_test_split(
    app_train_nzv,
    train_size=sample_frac,
    stratify=app_train_nzv[TARGET],
    random_state=42
)

print(f"Original Shape : {app_train_nzv.shape}")
print(f"Sample Shape   : {iv_sample.shape}")

print(f"Original Bad Rate : ", f"{app_train_nzv[TARGET].mean():.4f}")
print(f"Sample Bad Rate   : " f"{iv_sample[TARGET].mean():.4f}")

IV_THRESHOLD = 0.02
ID_COLS = ["SK_ID_CURR"]

feature_cols = [
    col
    for col in app_train_nzv.columns
    if col not in ID_COLS + [TARGET]
]
iv_report = []

for col in feature_cols:

    try:

        x = iv_sample[col]
        y = iv_sample[TARGET]

        optb = OptimalBinning(
            name=col,
            dtype="numerical" if pd.api.types.is_numeric_dtype(x) else "categorical"
        )

        optb.fit(x, y)

        iv = optb.binning_table.build()["IV"].sum()

        iv_report.append({
            "COLUMN_NAME": col,
            "IV": iv,
            "DROP_FLAG": "DROP" if iv < IV_THRESHOLD else "KEEP"
        })

    except Exception as e:
        print(e)
        iv_report.append({
            "COLUMN_NAME": col,
            "IV": np.nan,
            "DROP_FLAG": "DROP"
        })

iv_report = pd.DataFrame(iv_report).sort_values(
    "IV",
    ascending=False
)

print(f"Total Features      : {len(iv_report)}")
print(f"Features To Keep    : {(iv_report['DROP_FLAG'] == 'KEEP').sum()}")
print(f"Features To Drop    : {(iv_report['DROP_FLAG'] == 'DROP').sum()}")

iv_report.head(20)

Original Shape : (307511, 374)
Sample Shape   : (100000, 374)
Original Bad Rate :  0.0807
Sample Bad Rate   : 0.0807
Total Features      : 372
Features To Keep    : 312
Features To Drop    : 60


,COLUMN_NAME,IV,DROP_FLAG
368,EXT_SOURCE_2,0.664442,KEEP
259,EXT_SOURCE_3,0.661885,KEEP
190,EXT_SOURCE_1,0.288517,KEEP
274,DAYS_CREDIT_MEAN_bureau,0.257552,KEEP
279,DAYS_CREDIT_MEDIAN_bureau,0.255447,KEEP
261,DAYS_EMPLOYED,0.242070,KEEP
362,AMT_PAYMENT_MIN_inst_pmt,0.222682,KEEP
281,DAYS_CREDIT_UPDATE_MEDIAN_bureau,0.203382,KEEP
278,DAYS_CREDIT_UPDATE_MEAN_bureau,0.186331,KEEP
371,DAYS_BIRTH,0.185023,KEEP


In [27]:
keep_cols = iv_report.loc[
    iv_report["DROP_FLAG"] == "KEEP",
    "COLUMN_NAME"
]

app_train_iv = app_train_nzv[[PRIMARY_JOIN_KEY] + keep_cols.tolist() + [TARGET]]
app_test_iv = app_test_nzv[[PRIMARY_JOIN_KEY] + keep_cols.tolist()]

print(app_train_nzv.shape, "->", app_train_iv.shape)
print(app_test_nzv.shape, "->", app_test_iv.shape)

(307511, 374) -> (307511, 314)
(48744, 373) -> (48744, 313)


## 5. Categorical Variable Report
- Drop columns with highly diversed (High percent of unique values out of all)

In [28]:
cat_cols = app_train_iv.select_dtypes(
    include=["object", "category"]
).columns.tolist()

cat_report = []

for col in cat_cols:

    vc = app_train_iv[col].value_counts(
        dropna=False,
        normalize=True
    )

    cat_report.append({
        "COLUMN_NAME": col,
        "N_UNIQUE": app_train_iv[col].nunique(dropna=False),
        "MISSING_PCT": round(
            app_train_iv[col].isnull().mean() * 100,
            2
        ),
        "TOP_CATEGORY": vc.index[0],
        "TOP_CATEGORY_PCT": round(
            vc.iloc[0] * 100,
            2
        ),
        "RARE_CATEGORY_COUNT": (
            (app_train_iv[col]
             .value_counts(normalize=True, dropna=True) < 0.01)
            .sum()
        )
    })

cat_report = pd.DataFrame(cat_report).sort_values(
    "N_UNIQUE",
    ascending=False
)

print(f"Total Categorical Features : {len(cat_report)}")

cat_report.head(20)

Total Categorical Features : 26


,COLUMN_NAME,N_UNIQUE,MISSING_PCT,TOP_CATEGORY,TOP_CATEGORY_PCT,RARE_CATEGORY_COUNT
1,ORGANIZATION_TYPE,58,0.00,Business Entity Type 3,22.11,41
10,NAME_GOODS_CATEGORY_GET_MODE_prev_app,27,5.35,XNA,44.07,18
0,OCCUPATION_TYPE,19,31.35,NaN,31.35,6
4,PRODUCT_COMBINATION_GET_MODE_prev_app,18,5.35,POS household with interest,19.68,2
11,NAME_SELLER_INDUSTRY_GET_MODE_prev_app,12,5.35,XNA,36.43,4
17,CREDIT_TYPE_GET_MODE_bureau,12,14.31,Consumer credit,74.69,8
8,CODE_REJECT_REASON_GET_MODE_prev_app,10,5.35,XAP,87.61,6
23,CHANNEL_TYPE_GET_MODE_prev_app,9,5.35,Country-wide,38.60,2
3,NAME_INCOME_TYPE,8,0.00,Working,51.63,4
25,STATUS_GET_MODE_bureau_bal,8,70.01,NaN,70.01,4


## 6. Final Candidate Variable List
- Create drop funnel of each filteration step

In [29]:
drop_funnel = pd.DataFrame({
    "STAGE": [
        "Initial Features",
        "After Fill Rate",
        "After NZV",
        "After IV"
    ],
    "FEATURE_COUNT": [
        app_train_merged.shape[1] - 1,
        app_train_fr.shape[1] - 1,
        app_train_nzv.shape[1] - 1,
        app_train_iv.shape[1] - 1
    ]
})

drop_funnel["FEATURES_DROPPED"] = (
    drop_funnel["FEATURE_COUNT"]
    .shift(1)
    .sub(drop_funnel["FEATURE_COUNT"])
    .fillna(0)
    .astype(int)
)

drop_funnel["DROP_PCT"] = (
    drop_funnel["FEATURES_DROPPED"]
    / drop_funnel["FEATURE_COUNT"].shift(1)
    * 100
).fillna(0).round(2)

drop_funnel["SURVIVAL_PCT"] = (
    drop_funnel["FEATURE_COUNT"]
    / drop_funnel.loc[0, "FEATURE_COUNT"]
    * 100
).round(2)

print(drop_funnel)

              STAGE  FEATURE_COUNT  FEATURES_DROPPED  DROP_PCT  SURVIVAL_PCT
0  Initial Features            399                 0      0.00        100.00
1   After Fill Rate            391                 8      2.01         97.99
2         After NZV            373                18      4.60         93.48
3          After IV            313                60     16.09         78.45


## Save Selected Features

In [ ]:
if SAVE_FILES:
    app_train_iv.to_csv(os.path.join(ROW_FS_SAVE_PATH, "app_train_fs.csv"), index=False)
    app_test_iv.to_csv(os.path.join(ROW_FS_SAVE_PATH, "app_test_fs.csv"), index=False)